# Day 3: VideoMAE feature extraction for FineGym vault clips

**Runtime → Change runtime type → T4 GPU** before running. Then *Runtime → Run all*.

For every usable clip in `clips_available.csv` this extracts, with a frozen VideoMAE (Kinetics-400 fine-tuned):

| feature | shape | used for |
|---|---|---|
| `whole` | 768 | score prediction on the whole vault (Wed) |
| `whole_seq` | 8 × 768 | temporal sequence of the vault, for alignment / video-video comparison later |
| `stages` | 3 × 768 | approach / flight / landing, for stage-wise scores and explainable comparison |

Features are saved to Drive in small parts, so if Colab disconnects just **Run all** again: finished clips are skipped.

Expected time on a T4: ~5 min copying clips, ~15-25 min extraction for ~440 clips.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT      = Path('/content/drive/MyDrive/capstone')
CLIPS_DRV = ROOT / 'clips'
CSV       = ROOT / 'out' / 'clips_available.csv'
FEAT_DIR  = ROOT / 'features' / 'videomae_k400'
FEAT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL     = Path('/content/clips')          # local copy: reading from Drive directly is slow

assert CSV.exists(), f'missing {CSV}'
assert CLIPS_DRV.exists(), f'missing {CLIPS_DRV}'
print('ok')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ok


## 2. Load the clip list (train + test only) and copy clips locally

In [4]:
import pandas as pd, shutil, os
df = pd.read_csv(CSV, dtype={'vault_code': str, 'bib': str})
df = df[df.split.isin(['train', 'test'])].reset_index(drop=True)
df['clip_id'] = df.video_id + '__' + df.event_key
print(df.split.value_counts().to_dict(), '| clips without stages:', df.segment_key.isna().sum())

LOCAL.mkdir(exist_ok=True)
copied = 0
for f in df.clip_file:
    dst = LOCAL / f
    if not dst.exists():
        shutil.copy(CLIPS_DRV / f, dst); copied += 1
print('copied', copied, '| local clips:', len(list(LOCAL.glob('*.mp4'))))

{'train': 260, 'test': 177} | clips without stages: 29
copied 437 | local clips: 437


## 3. Video reading and window sampling
Stage times in the CSV are relative to the vault start; each downloaded file starts `stage_offset` (0.5 s) earlier, so the offset is added.

In [5]:
import json
import math

import cv2
import numpy as np

N_FRAMES = 16  # VideoMAE input length


def read_clip(path, short_side=256):
    """Read every frame of a short clip into memory (RGB, resized so the short side = short_side)."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        h, w = f.shape[:2]
        s = short_side / min(h, w)
        f = cv2.resize(f, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps


def sample_window(frames, fps, t0, t1, n=N_FRAMES):
    """n frames evenly spaced in [t0, t1) seconds (centre of n equal bins); clamps to the clip."""
    last = len(frames) - 1
    t1 = max(t1, t0 + 1e-3)
    times = [t0 + (i + 0.5) * (t1 - t0) / n for i in range(n)]
    idx = [min(last, max(0, int(math.floor(t * fps)))) for t in times]
    return [frames[i] for i in idx], idx


def windows_for_row(row):
    """(whole, [stage1, stage2, stage3] or None) windows in seconds inside the downloaded file.
    Annotation times are relative to the vault start; the file starts `stage_offset` s earlier."""
    off = float(row["stage_offset"])
    whole = (off, off + float(row["duration"]))
    stages = None
    raw = row.get("stages")
    if isinstance(raw, str) and raw.strip().startswith("["):
        st = json.loads(raw)
        if len(st) == 3:
            stages = [(off + a, off + b) for a, b in st]
    return whole, stages

## 4. Model
Frozen VideoMAE-Base fine-tuned on Kinetics-400. The 768-d feature is the model's own pooled representation (mean over tokens + `fc_norm`), i.e. exactly what its classifier sees.

In [6]:
import torch, numpy as np
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

CKPT = 'MCG-NJU/videomae-base-finetuned-kinetics'
DEV = 'cuda'
proc  = VideoMAEImageProcessor.from_pretrained(CKPT)
model = VideoMAEForVideoClassification.from_pretrained(CKPT, torch_dtype=torch.float16).to(DEV).eval()
T_TOK = N_FRAMES // model.config.tubelet_size          # 8 temporal token groups

@torch.no_grad()
def embed(videos):
    """videos: list of clips, each a list of N_FRAMES RGB frames -> (pooled [B,768], seq [B,8,768])"""
    px = proc(videos, return_tensors='pt')['pixel_values'].to(DEV, torch.float16)
    h = model.videomae(px).last_hidden_state                       # [B, 8*196, 768]
    pooled = model.fc_norm(h.mean(1)) if model.fc_norm is not None else h.mean(1)  # [B, 768]
    seq = h.view(h.shape[0], T_TOK, -1, h.shape[-1]).mean(2)        # [B, 8, 768] (time-major tokens)
    return pooled.float().cpu().numpy(), seq.float().cpu().numpy()

# smoke test on one clip
r = df.iloc[0]
fr, fps = read_clip(LOCAL / r.clip_file)
p, s = embed([sample_window(fr, fps, *windows_for_row(r)[0])[0]])
print(r.clip_id, len(fr), 'frames @', fps, 'fps ->', p.shape, s.shape)

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.9k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

1JsRXIoR3C0__E_000598_000605 202 frames @ 25.0 fps -> (1, 768) (1, 8, 768)


## 5. Extract (resumable)
Saves `part_XXX.npz` every 25 clips. Safe to re-run after a disconnect.

In [7]:
import glob, time
from tqdm.auto import tqdm

done = set()
for f in sorted(FEAT_DIR.glob('part_*.npz')):
    done.update(np.load(f, allow_pickle=True)['ids'].tolist())
todo = df[~df.clip_id.isin(done)]
print(f'{len(done)} already done, {len(todo)} to go')

SAVE_EVERY = 25
buf = {k: [] for k in ['ids', 'whole', 'whole_seq', 'stages', 'has_stages', 'n_frames', 'fps']}
part = len(list(FEAT_DIR.glob('part_*.npz')))
errors = []

def flush():
    global part, buf
    if not buf['ids']: return
    np.savez_compressed(FEAT_DIR / f'part_{part:03d}.npz',
        ids=np.array(buf['ids']), whole=np.stack(buf['whole']).astype(np.float16),
        whole_seq=np.stack(buf['whole_seq']).astype(np.float16),
        stages=np.stack(buf['stages']).astype(np.float16),
        has_stages=np.array(buf['has_stages']), n_frames=np.array(buf['n_frames']), fps=np.array(buf['fps']))
    part += 1
    buf = {k: [] for k in buf}

t0 = time.time()
for r in tqdm(todo.itertuples(), total=len(todo)):
    try:
        fr, fps = read_clip(LOCAL / r.clip_file)
        if len(fr) < N_FRAMES:
            raise ValueError(f'only {len(fr)} frames')
        whole, stages = windows_for_row(r._asdict())
        wins = [whole] + (stages or [])
        pooled, seq = embed([sample_window(fr, fps, a, b)[0] for a, b in wins])
        buf['ids'].append(r.clip_id)
        buf['whole'].append(pooled[0]); buf['whole_seq'].append(seq[0])
        buf['stages'].append(pooled[1:4] if stages else np.full((3, pooled.shape[1]), np.nan, np.float32))
        buf['has_stages'].append(stages is not None)
        buf['n_frames'].append(len(fr)); buf['fps'].append(fps)
    except Exception as e:
        errors.append((r.clip_id, repr(e)[:150]))
    if len(buf['ids']) >= SAVE_EVERY:
        flush()
flush()
print(f'done in {(time.time()-t0)/60:.1f} min | errors: {len(errors)}')
for e in errors[:10]: print('  ', e)

0 already done, 437 to go


  0%|          | 0/437 [00:00<?, ?it/s]

done in 9.2 min | errors: 0


## 6. Merge into one file

In [8]:
parts = [np.load(f, allow_pickle=True) for f in sorted(FEAT_DIR.glob('part_*.npz'))]
F = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
# drop duplicates (possible if a run was interrupted mid-save) and keep CSV order
_, first = np.unique(F['ids'], return_index=True)
F = {k: v[np.sort(first)] for k, v in F.items()}
order = {c: i for i, c in enumerate(df.clip_id)}
idx = np.argsort([order[c] for c in F['ids']])
F = {k: v[idx] for k, v in F.items()}

out = ROOT / 'features' / 'videomae_k400_all.npz'
np.savez_compressed(out, **F)
meta = df.set_index('clip_id').loc[F['ids']].reset_index()
meta.to_csv(ROOT / 'features' / 'videomae_k400_index.csv', index=False)
print(out, {k: v.shape for k, v in F.items()})

/content/drive/MyDrive/capstone/features/videomae_k400_all.npz {'ids': (437,), 'whole': (437, 768), 'whole_seq': (437, 8, 768), 'stages': (437, 3, 768), 'has_stages': (437,), 'n_frames': (437,), 'fps': (437,)}


## 7. Sanity checks
1. No NaNs in `whole`; stage features present where expected.
2. **Do the features know *which* vault was done?** Nearest-neighbour (cosine) from each test clip to the train clips: how often does the neighbour have the same `vault_code`? Compare with always guessing the most common code. High = features capture action identity (good for retrieval, and the reason Total/D are "easy").
3. **Do stages differ?** Average cosine similarity between approach / flight / landing features of the same clip vs. the same stage across clips.

In [9]:
W = F['whole'].astype(np.float32)
print('NaNs in whole:', np.isnan(W).sum(), '| clips with stages:', F['has_stages'].sum(), '/', len(W))

def norm(x): return x / np.linalg.norm(x, axis=-1, keepdims=True)
tr, te = (meta.split == 'train').values, (meta.split == 'test').values
sim = norm(W[te]) @ norm(W[tr]).T
nn_code = meta.vault_code.values[tr][sim.argmax(1)]
acc = (nn_code == meta.vault_code.values[te]).mean()
base = (meta.vault_code.values[te] == meta.vault_code[tr].mode()[0]).mean()
print(f'1-NN vault_code accuracy (test -> train): {acc:.3f}  vs majority-class baseline {base:.3f}')

S = norm(F['stages'][F['has_stages']].astype(np.float32))           # [n, 3, 768]
within = np.mean([S[:, i] @ S[:, j].T for i, j in [(0,1),(1,2),(0,2)]], axis=0).diagonal().mean()
across = np.mean([(S[:, k] @ S[:, k].T)[np.triu_indices(len(S), 1)].mean() for k in range(3)])
print(f'cosine: different stages of the same clip {within:.3f} | same stage across clips {across:.3f}')

NaNs in whole: 0 | clips with stages: 408 / 437
1-NN vault_code accuracy (test -> train): 0.130  vs majority-class baseline 0.260
cosine: different stages of the same clip 0.764 | same stage across clips 0.768


In [10]:
summary = f'''DAY 3 FEATURE EXTRACTION (VideoMAE-Base, Kinetics-400, frozen)
clips: {len(W)} (train {tr.sum()}, test {te.sum()}) | with stage features: {F['has_stages'].sum()}
errors: {len(errors)}
1-NN vault_code accuracy test->train: {acc:.3f} (majority baseline {base:.3f})
cosine different stages same clip: {within:.3f} | same stage across clips: {across:.3f}
'''
(ROOT / 'features' / 'day3_summary.txt').write_text(summary)
print(summary)

DAY 3 FEATURE EXTRACTION (VideoMAE-Base, Kinetics-400, frozen)
clips: 437 (train 260, test 177) | with stage features: 408
errors: 0
1-NN vault_code accuracy test->train: 0.130 (majority baseline 0.260)
cosine different stages same clip: 0.764 | same stage across clips: 0.768

